**PROYECTO**       : MIGRACION SANDBOX  
**NOMBRE**         : nb_td_pago_servicio.ipynb  
**TABLA DESTINO**  : mb_gold_prod.inof.td_pago_servicio  
**TABLA FUENTE**   : mb_bronze_prod.det.de_wm_tabla_pago_servicio_df  
**OBJETIVO**       : Poblar la tabla de pagos de servicio desde el Data Entry  
**TIPO**           : PYTHON  
**REPROCESABLE**   : SI  
**OBSERVACION**    : NA  
**SCHEDULER**      : NA  
**JOB**            : NA
| VERSION | DESARROLLADOR | PROVEEDOR | PO | FECHA | DESCRIPCION |
|---------|---------------|-----------|----|-------|-------------|
| 1.0 | Esteban Offer | MIBANCO | Willyan Soto | 2026-09-11 | Creacion de proceso |

## 1. Librerias y dependencias

In [0]:
import logging
import time

from pyspark.sql import functions as F

## 2. Funciones de transformacion

In [0]:
def read_pagos(tabla):
    """Lee y proyecta los pagos de servicio del Data Entry."""
    return spark.table(tabla).select(
        F.col("Referencia").cast("string").alias("cod_pago_servicio"),
        F.to_timestamp(F.col("Fecha"), "d/MM/yyyy HH:mm").alias("fec_pago_servicio"),
        F.col("Empresa").cast("string").alias("nom_empresa_servicio"),
        F.col("Monto").cast("decimal(18,2)").alias("mto_servicio_origen"),
        F.col("Documento_dac").alias("nro_documento_cliente_dac"),
        F.col("_ingestion_time"),
        F.col("_processing_time"),
    )


def add_estado_conciliacion(df_pagos):
    """Marca el estado de conciliacion del pago segun el monto."""
    return df_pagos.withColumn(
        "des_estado_conciliacion",
        F.when(F.col("mto_servicio_origen") > 0, F.lit("CONCILIADO"))
        .otherwise(F.lit("PENDIENTE")),
    )

## 3. Parametros de entrada

In [0]:
dbutils.widgets.text("ambiente", "dev")
dbutils.widgets.text("fechaproceso", "")

var_ambiente = dbutils.widgets.get("ambiente")
var_fechaproceso = dbutils.widgets.get("fechaproceso")

logger = logging.getLogger("TD_PAGO_SERVICIO")
logger.setLevel(logging.INFO)
ini_proceso = time.perf_counter()
logger.info("Inicio del proceso. ambiente=%s fechaproceso=%s",
            var_ambiente, var_fechaproceso)

## 4. Constantes y variables

In [0]:
TBL_PAGO_SRC = f"mb_bronze_{var_ambiente}.det.de_wm_tabla_pago_servicio_df"
TBL_EMPRESA_SRC = f"mb_silver_{var_ambiente}.mmff.m_empresa_servicio"
TBL_PAGO_FIN = f"mb_gold_{var_ambiente}.inof.td_pago_servicio"

## 5. Logica principal

In [0]:
ini_etapa = time.perf_counter()

df_pagos = read_pagos(TBL_PAGO_SRC)
df_empresas = spark.table(TBL_EMPRESA_SRC).select(
    "nom_empresa_servicio", "cod_empresa_servicio"
)

df_pagos_enriquecido = add_estado_conciliacion(
    df_pagos.join(df_empresas, "nom_empresa_servicio", "left")
)

logger.info("Tiempo de transformacion: %.2f segundos",
            time.perf_counter() - ini_etapa)

## 6. Escritura

In [0]:
try:
    (
        df_pagos_enriquecido
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(TBL_PAGO_FIN)
    )
except Exception as exc:
    logger.error("Fallo la escritura en %s: %s", TBL_PAGO_FIN, exc)
    raise

logger.info("Fin del proceso. Duracion total: %.2f segundos",
            time.perf_counter() - ini_proceso)